# Optimized Rain Removal for Object Detection

**Key Insight**: For object detection, we need to PRESERVE edges and texture, not remove them!

This pipeline:
- Enhances contrast and visibility
- Reduces rain streaks WITHOUT destroying object boundaries
- Preserves high-frequency details crucial for detection
- Processes 10-20x faster than MCA approach

**Why the old approach failed**:
- Bilateral filtering over-smoothed object edges
- Dictionary learning removed legitimate edge features
- Complete rain atom removal destroyed texture
- Color scaling distorted appearances

In [7]:
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import os
import time
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

## Core Rain Removal Functions

Strategy:
1. **CLAHE** - Adaptive contrast enhancement for better visibility
2. **Guided Filter** - Edge-preserving smoothing (less aggressive than bilateral)
3. **Directional Rain Suppression** - Target vertical streaks specifically
4. **Edge Preservation** - Sharpen boundaries for better detection
5. **Gamma Correction** - Brighten dark regions without saturation

In [22]:
import cv2
import numpy as np
from PIL import Image
# We need pywt for the Discrete Wavelet Transform
# You will need to install it: pip install PyWavelets
import pywt

# --- Existing Guided Filter (Adapted for potential multi-channel input) ---

def guided_filter_gray(I, p, r, eps):
    """
    Standard Guided Filter implementation for a single channel.
    Input/Output: single channel (2D numpy array), np.float32 or np.float64.
    I: guidance image, p: input image, r: radius, eps: regularization.
    """
    # Ensure all inputs are float64 for high precision calculations
    I = I.astype(np.float64)
    p = p.astype(np.float64)
    
    mean_I = cv2.boxFilter(I, cv2.CV_64F, (r, r))
    mean_p = cv2.boxFilter(p, cv2.CV_64F, (r, r))
    mean_Ip = cv2.boxFilter(I * p, cv2.CV_64F, (r, r))
    cov_Ip = mean_Ip - mean_I * mean_p
    
    mean_II = cv2.boxFilter(I * I, cv2.CV_64F, (r, r))
    var_I = mean_II - mean_I * mean_I
    
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    
    mean_a = cv2.boxFilter(a, cv2.CV_64F, (r, r))
    mean_b = cv2.boxFilter(b, cv2.CV_64F, (r, r))
    
    # Equation (31) and (35) use I(x) as the guidance image for the Base Layer B(x)
    q = mean_a * I + mean_b
    
    # No resize needed if p and I are the same size, which they should be for base layer decomp
    
    return q.astype(p.dtype)


def soft_threshold(coeff, threshold):
    """
    Soft Thresholding: D'(C, lambda) = sgn(C) * max(0, |C| - lambda)
    This is applied to the detail coefficients (CH, CV, CD).
    """
    return np.sign(coeff) * np.maximum(0, np.abs(coeff) - threshold)


def remove_rain_wavelet(
    img,
    r=40,            # Guided Filter radius (for decomposition)
    eps=0.001,       # Guided Filter regularization (for decomposition)
    wavelet='haar',  # Wavelet type (e.g., 'haar', 'db1', 'bior1.3')
    level=1,         # DWT decomposition level
    threshold=0.05   # Soft Thresholding value (lambda)
):
    """
    Rain removal using Guided Filter for decomposition and DWT soft-thresholding.
    Based on the 'Alternative Method' description.
    """
    if isinstance(img, Image.Image):
        img = np.array(img)

    if img.shape[-1] == 4:
        img = img[:, :, :3]

    # Normalize to [0, 1] for processing
    img_float = img.astype(np.float32) / 255.0
    
    # Convert to YCrCb space for processing on the luminance channel (Y)
    # The paper description implies processing on the primary image layer,
    # often Y or V, which is most standard for rain/denoising.
    ycbcr = cv2.cvtColor(img_float, cv2.COLOR_RGB2YCrCb)
    Y = ycbcr[:, :, 0]

    # --- 1. Guided Filtering for Image Decomposition (Equations 31 & 32) ---
    
    # Base Layer B(x) = GuidedFilter(I(x), I(x))
    # Note: Use a larger radius (r) and small epsilon (eps) for a smooth base layer
    B = guided_filter_gray(Y, Y, r=r, eps=eps)
    
    # Detail Layer D(x) = I(x) - B(x)
    D = Y - B

    # --- 2. Discrete Wavelet Transform and Thresholding (Equations 33 & 34) ---
    
    # 2D DWT on the Detail Layer D(x) -> {CA, CH, CV, CD}
    # pywt.wavedec2 returns a tuple: (cA, (cH, cV, cD), (cH2, cV2, cD2), ...)
    coeffs = pywt.wavedec2(D, wavelet, level=level)
    
    cA = coeffs[0]
    
    # Apply Soft Thresholding to the directional detail subbands (CH, CV, CD, etc.)
    # Rain streaks are predominantly high-frequency and directional.
    thresholded_detail_coeffs = []
    
    # Iterate through detail levels
    for detail_level in coeffs[1:]:
        # detail_level is a tuple: (cH, cV, cD)
        cH, cV, cD = detail_level
        
        # Apply soft thresholding to each detail subband
        cH_t = soft_threshold(cH, threshold)
        cV_t = soft_threshold(cV, threshold)
        cD_t = soft_threshold(cD, threshold)
        
        thresholded_detail_coeffs.append((cH_t, cV_t, cD_t))

    # Reconstruct cleaned detail layer D'(x) using Inverse DWT
    coeffs_clean = [cA] + thresholded_detail_coeffs
    D_prime = pywt.waverec2(coeffs_clean, wavelet)
    
    # Handle potential size mismatch from DWT/IDWT due to padding (common issue)
    if D_prime.shape != Y.shape:
        D_prime = cv2.resize(D_prime, (Y.shape[1], Y.shape[0]), interpolation=cv2.INTER_LINEAR)


    # --- 3. Reconstruction (Equation 35) ---
    
    # J(x) = B(x) + D'(x)
    J = B + D_prime
    
    # Clamp and put back into YCrCb
    ycbcr[:, :, 0] = np.clip(J, 0.0, 1.0)
    
    # Convert back to RGB
    img_clean = cv2.cvtColor(ycbcr, cv2.COLOR_YCrCb2RGB)

    # Re-scale to [0, 255] and convert to uint8
    return np.clip(img_clean * 255.0, 0, 255).astype(np.uint8)

# Example usage (assuming you have an image loaded as 'input_img'):
# from PIL import Image
# input_img = Image.open("your_rainy_image.jpg")
# output_img = remove_rain_wavelet(input_img)
# Image.fromarray(output_img).save("denoised_wavelet.png")

# Configuration for this method (can be passed to the function)
CONFIG = {
    "r": 40,
    "eps": 0.001,
    "wavelet": 'db1',
    "level": 1,
    "threshold": 0.05
}

In [10]:
import cv2
import numpy as np
from PIL import Image

def remove_rain_mca_like(img,
                         bilateral_d=9,
                         sigma_color=50,
                         sigma_space=7,
                         hf_thresh=0.02,
                         directionality_thresh=0.6,
                         gamma=1.1,
                         clahe_clip=2.0,
                         sharpen_amount=1.2):
    """
    MCA-inspired rain removal using:
    - LF/HF decomposition (bilateral filter)
    - Directional energy detection (rain streak prior)
    - Post-enhancement for contrast, brightness, and sharpness
    """

    # --- Handle PIL Images ---
    if isinstance(img, Image.Image):
        img = np.array(img)

    img = img.astype(np.float32) / 255.0

    # ====================================================
    # 1. Low–High Frequency Decomposition
    # ====================================================
    lf = cv2.bilateralFilter(img, bilateral_d, sigma_color, sigma_space)
    hf = img - lf

    # ====================================================
    # 2. High-Frequency Grayscale
    # ====================================================
    hf_gray = cv2.cvtColor(np.abs(hf), cv2.COLOR_RGB2GRAY)

    # ====================================================
    # 3. Directional Filtering (Rain Prior)
    # ====================================================
    k_v = cv2.getGaborKernel((9, 9), 3, np.pi / 2, 8, 0.5)
    k_d = cv2.getGaborKernel((9, 9), 3, np.pi / 4, 8, 0.5)

    resp_v = cv2.filter2D(hf_gray, -1, k_v)
    resp_d = cv2.filter2D(hf_gray, -1, k_d)

    directional_energy = np.maximum(resp_v, resp_d)

    # ====================================================
    # 4. Rain Mask (MCA-style Atom Selection)
    # ====================================================
    rain_mask = (
        (hf_gray > hf_thresh) &
        (directional_energy > directionality_thresh * directional_energy.max())
    ).astype(np.float32)

    rain_mask_3c = np.repeat(rain_mask[:, :, None], 3, axis=2)

    # ====================================================
    # 5. Remove Rain Atoms (HF_nonrain)
    # ====================================================
    hf_nonrain = hf * (1 - rain_mask_3c)

    # ====================================================
    # 6. Reconstruct Image
    # ====================================================
    clean = lf + hf_nonrain
    clean = np.clip(clean, 0, 1)

    # ====================================================
    # 7. Contrast Enhancement (CLAHE on L channel)
    # ====================================================
    lab = cv2.cvtColor((clean * 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    clean = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32) / 255.0

    # ====================================================
    # 8. Brightness Correction (Gamma)
    # ====================================================
    clean = np.power(clean, 1.0 / gamma)

    # ====================================================
    # 9. Sharpness Recovery (Unsharp Mask)
    # ====================================================
    blurred = cv2.GaussianBlur(clean, (0, 0), 1.5)
    clean = cv2.addWeighted(clean, sharpen_amount, blurred, 1 - sharpen_amount, 0)

    clean = np.clip(clean, 0, 1)

    return (clean * 255).astype(np.uint8)


## Batch Processing - Fast Sequential

In [11]:
import os
import time
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import numpy as np
import shutil # Import shutil for directory removal

# Assuming CONFIG and remove_rain_optimized are defined elsewhere

def process_directory(input_dir, output_dir, show_progress=True):
    """
    Process all images in a directory.
    Fast sequential processing with progress bar.
    
    NOTE: The output_dir is CLEARED before processing.
    """

    
    # --- NEW: Clear existing output directory ---
    if os.path.exists(output_dir):
        print(f"Clearing existing output directory: {output_dir}")
        shutil.rmtree(output_dir)
    # --- END NEW ---
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Find images
    exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')
    files = sorted([f for f in Path(input_dir).iterdir() 
                    if f.suffix.lower() in exts])
    
    if not files:
        print(f"No images found in {input_dir}")
        return
    
    print(f"{'='*80}")
    print(f"{'OPTIMIZED RAIN REMOVAL - BATCH PROCESSING':^80}")
    print(f"{'='*80}")
    print(f"Input:   {input_dir}")
    print(f"Output:  {output_dir}")
    print(f"Images:  {len(files)}")
    print(f"{'='*80}\n")
    
    times = []
    total_start = time.time()
    
    iterator = tqdm(files, desc="Processing") if show_progress else files
    
    for img_path in iterator:
        try:
            # Load
            img = Image.open(img_path).convert('RGB')
            
            # Process
            start = time.time()
            # This line assumes 'remove_rain_optimized' is available
            clean = remove_rain_mca_like(img) 
            elapsed = time.time() - start
            times.append(elapsed)
            
            # Save
            output_path = Path(output_dir) / img_path.name
            # This line assumes 'clean' is a numpy array (as used by Image.fromarray)
            Image.fromarray(clean).save(output_path, quality=95) 
            
            if not show_progress:
                print(f"✓ {img_path.name} ({elapsed:.2f}s)")
                
        except Exception as e:
            print(f"✗ {img_path.name}: {e}")
    
    total_time = time.time() - total_start
    
    print(f"\n{'='*80}")
    print(f"{'✓ COMPLETE!':^80}")
    print(f"{'='*80}")
    print(f"Processed: {len(times)}/{len(files)} images")
    print(f"Total:     {total_time:.1f}s ({total_time/60:.2f} min)")
    print(f"Average:   {np.mean(times):.2f}s/image")
    print(f"Fastest:   {np.min(times):.2f}s")
    print(f"Slowest:   {np.max(times):.2f}s")
    print(f"Throughput: {len(times)/total_time:.1f} images/sec")
    print(f"Output:    {output_dir}")
    print(f"{'='*80}\n")


# Example of how to run the batch processing
# INPUT_DIR = "images_pre_processed"
# OUTPUT_DIR = "inputs_images_processed"

# # Ensure CONFIG is defined before running
# # CONFIG = {...} 

# # Run batch processing
# process_directory(INPUT_DIR, OUTPUT_DIR, config=CONFIG, show_progress=True)

# Run batch processing
INPUT_DIR = "images_pre_processed"
OUTPUT_DIR = "inputs_images_processed_mca_2"

process_directory(INPUT_DIR, OUTPUT_DIR, show_progress=True)

Clearing existing output directory: inputs_images_processed_mca_2
                   OPTIMIZED RAIN REMOVAL - BATCH PROCESSING                    
Input:   images_pre_processed
Output:  inputs_images_processed_mca_2
Images:  99



Processing: 100%|██████████| 99/99 [02:34<00:00,  1.56s/it]


                                  ✓ COMPLETE!                                   
Processed: 99/99 images
Total:     154.5s (2.57 min)
Average:   1.44s/image
Fastest:   0.43s
Slowest:   4.54s
Throughput: 0.6 images/sec
Output:    inputs_images_processed_mca_2



## Batch Processing - Parallel (for large datasets)

## Parameter Tuning Guide

If results aren't optimal, adjust these parameters:

### For Better Detection Scores:
1. **Increase `sharpen_amount`** (0.4 → 0.6): Enhances object edges
2. **Decrease `guided_radius`** (4 → 3): Preserves more detail
3. **Increase `clahe_clip_limit`** (2.5 → 3.0): Better contrast in dark areas

### For Heavy Rain:
1. **Decrease `rain_threshold`** (0.08 → 0.05): Detect more rain
2. **Increase `rain_kernel_length`** (15 → 19): Better streak detection
3. **Increase `gamma`** (1.1 → 1.2): Brighten dark regions

### For Speed:
1. **Decrease `clahe_tile_size`** (16 → 8): Faster CLAHE
2. **Decrease `guided_radius`** (4 → 3): Faster filtering
3. Use **parallel processing** for many images

## Compare with Original Approach

In [ ]:
# Visual comparison of processing speed
test_images = [f for f in Path(INPUT_DIR).glob('*.jpg')][:5]

if test_images:
    print("Speed comparison on 5 sample images:\n")
    
    for img_path in test_images:
        img = Image.open(img_path).convert('RGB')
        
        start = time.time()
        clean = remove_rain_optimized(img, **CONFIG)
        elapsed = time.time() - start
        
        print(f"{img_path.name:30s} {img.size[0]:4d}x{img.size[1]:4d}  {elapsed:5.2f}s  ({img.size[0]*img.size[1]/elapsed/1e6:.1f} MP/s)")
    
    print("\nOld MCA approach: 20-60s per image")
    print("New approach: 0.5-3s per image")
    print("Speedup: 10-30x faster! 🚀")